# 1. Import Libraries

In [16]:
# Import core libraries for data manipulation
import numpy as np
import pandas as pd

# Visualization libraries (used for EDA & checking data issues)
import matplotlib.pyplot as plt
import seaborn as sns

# Libraries for text cleaning and processing
import re
import string

# Date handling
from datetime import datetime

# Feature scaling (used before ML models)
from sklearn.preprocessing import StandardScaler

# Ignore warnings to keep output clean
import warnings

# 2. Load Data

In [17]:
# Load dataset from CSV file
df = pd.read_csv("netflix_titles.csv")

# Display first 5 rows to understand structure
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [18]:
df.shape

(8807, 12)

In [19]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

In [20]:
df.tail()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."
8806,s8807,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,"March 2, 2019",2015,TV-14,111 min,"Dramas, International Movies, Music & Musicals",A scrappy but poor boy worms his way into a ty...


# 3. Basic Cleaning

In [21]:
df.duplicated()#checking condition FYI

0       False
1       False
2       False
3       False
4       False
        ...  
8802    False
8803    False
8804    False
8805    False
8806    False
Length: 8807, dtype: bool

In [22]:
#checking duplicates
df[df.duplicated()]#dataframe showing duplicates

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description


In [23]:
#df[df.duplicated(keep=False)] # FYI

In [24]:
#dropping duplicates
# df = df.drop_duplicates()

In [25]:
#df.shape

In [26]:
# check missing values
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.9 MB


In [28]:
# Fill missing categorical values with 'Unknown'
# This is preferred in industry instead of dropping rows
for col in ['director', 'cast', 'country']:
    df[col] = df[col].fillna('Unknown')

In [29]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         3
listed_in        0
description      0
dtype: int64

In [30]:
# Fill missing ratings with most frequent value (mode)
# Keeps dataset consistent without losing data
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

In [31]:
df['rating'].mode() #FYI

0    TV-MA
Name: rating, dtype: str

In [32]:
df['rating'].describe() #FYI

count      8807
unique       17
top       TV-MA
freq       3211
Name: rating, dtype: object

In [33]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         3
listed_in        0
description      0
dtype: int64

# 4. Date Processing

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      8807 non-null   str  
 4   cast          8807 non-null   str  
 5   country       8807 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8807 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.9 MB


In [35]:
# Convert 'date_added' column to datetime format
# errors='coerce' converts invalid values to NaT instead of crashing
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

In [36]:
df.dtypes

show_id                    str
type                       str
title                      str
director                   str
cast                       str
country                    str
date_added      datetime64[us]
release_year             int64
rating                     str
duration                   str
listed_in                  str
description                str
dtype: object

In [37]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [38]:
# Extract useful features from date
df['year_added'] = df['date_added'].dt.year   # Year content was added
df['month_added'] = df['date_added'].dt.month # Month added
df['day_added'] = df['date_added'].dt.day     # Day added

In [39]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,day_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,25.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,24.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,24.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,24.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,24.0


In [40]:

df.dtypes

show_id                    str
type                       str
title                      str
director                   str
cast                       str
country                    str
date_added      datetime64[us]
release_year             int64
rating                     str
duration                   str
listed_in                  str
description                str
year_added             float64
month_added            float64
day_added              float64
dtype: object

In [41]:
#“Pandas converts integers to float when there are missing values, because float can store NaN but int cannot.”

# 5. Feature Engineering

Feature engineering is the process of transforming raw data into structured variables that improve interpretability, enable deeper analysis, and support better decision-making.

### 5.1 Duration Feature

In [42]:
# Function to extract numeric value from duration column
# Example: "90 min" → 90, "2 Seasons" → 2
def extract_duration(x):
    if pd.isna(x):
        return np.nan  # Handle missing values safely
    
    return int(x.split()[0])  # Extract number part

# Apply function to create numeric duration feature
df['duration_int'] = df['duration'].apply(extract_duration)

In [43]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,day_added,duration_int
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,25.0,90.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,24.0,2.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,24.0,1.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,24.0,1.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,24.0,2.0


In [44]:
df['type'].unique()

<ArrowStringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

### 5.2 Content Type Encoding

In [45]:
# Convert 'type' column into binary feature
# Movie = 1, TV Show = 0 (useful for ML models)
df['is_movie'] = df['type'].apply(lambda x: 1 if x == 'Movie' else 0)

In [46]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,day_added,duration_int,is_movie
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,25.0,90.0,1
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,24.0,2.0,0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,24.0,1.0,0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,24.0,1.0,0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,24.0,2.0,0


In [47]:
df['country'].value_counts()

country
United States                                    2818
India                                             972
Unknown                                           831
United Kingdom                                    419
Japan                                             245
                                                 ... 
Russia, Spain                                       1
Croatia, Slovenia, Serbia, Montenegro               1
Japan, Canada                                       1
United States, France, South Korea, Indonesia       1
United Arab Emirates, Jordan                        1
Name: count, Length: 749, dtype: int64

In [48]:

df['country'].value_counts()


country
United States                                    2818
India                                             972
Unknown                                           831
United Kingdom                                    419
Japan                                             245
                                                 ... 
Russia, Spain                                       1
Croatia, Slovenia, Serbia, Montenegro               1
Japan, Canada                                       1
United States, France, South Korea, Indonesia       1
United Arab Emirates, Jordan                        1
Name: count, Length: 749, dtype: int64

In [49]:
df['listed_in'].value_counts()


listed_in
Dramas, International Movies                        362
Documentaries                                       359
Stand-Up Comedy                                     334
Comedies, Dramas, International Movies              274
Dramas, Independent Movies, International Movies    252
                                                   ... 
Comedies, Cult Movies, LGBTQ Movies                   1
Action & Adventure, Comedies, Horror Movies           1
Classic & Cult TV, Crime TV Shows, TV Dramas          1
Action & Adventure, Documentaries, Sports Movies      1
Cult Movies, Dramas, Thrillers                        1
Name: count, Length: 514, dtype: int64

### 5.3 Multi-Value Column Features

In [50]:
# Count number of actors in cast column
# More actors may indicate big production
df['cast_count'] = df['cast'].apply(lambda x: len(x.split(',')))

# Count number of genres
df['genre_count'] = df['listed_in'].apply(lambda x: len(x.split(',')))

# Count number of countries involved
df['country_count'] = df['country'].apply(lambda x: len(x.split(',')))

In [51]:
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,day_added,duration_int,is_movie,cast_count,genre_count,country_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,25.0,90.0,1,1,1,1
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,24.0,2.0,0,19,3,1
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,24.0,1.0,0,9,3,1
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,24.0,1.0,0,1,2,1
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,24.0,2.0,0,8,3,1


In [52]:
df['country_count'].unique()

array([ 1,  6,  2,  3,  4,  5,  8,  7, 12, 10])

In [53]:
df[df['country_count']==12]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,day_added,duration_int,is_movie,cast_count,genre_count,country_count
6233,s6234,Movie,Barbecue,Matthew Salleh,Unknown,"Australia, Armenia, Japan, Jordan, Mexico, Mon...",2017-08-15,2017,TV-MA,101 min,"Documentaries, International Movies",A blend of cultural nuance and mesmerizing tec...,2017.0,8.0,15.0,101.0,1,1,2,12


In [54]:
df['country_count'].value_counts()

country_count
1     7487
2      873
3      273
4      114
5       37
6       14
7        5
8        2
12       1
10       1
Name: count, dtype: int64

### 5.4 Primary Entity Extraction

In [55]:
# Extract first country (main production country)
df['primary_country'] = df['country'].apply(lambda x: x.split(',')[0])

# Extract main genre
df['primary_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])

# Extract main director
df['primary_director'] = df['director'].apply(lambda x: x.split(',')[0])

In [56]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,month_added,day_added,duration_int,is_movie,cast_count,genre_count,country_count,primary_country,primary_genre,primary_director
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,9.0,25.0,90.0,1,1,1,1,United States,Documentaries,Kirsten Johnson
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,9.0,24.0,2.0,0,19,3,1,South Africa,International TV Shows,Unknown
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,9.0,24.0,1.0,0,9,3,1,Unknown,Crime TV Shows,Julien Leclercq
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,9.0,24.0,1.0,0,1,2,1,Unknown,Docuseries,Unknown
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,9.0,24.0,2.0,0,8,3,1,India,International TV Shows,Unknown


### 5.5 Business Features

In [57]:
c = datetime.now().year
print(c)#FYI

2026


In [58]:
df['release_year'].unique()

array([2020, 2021, 1993, 2018, 1996, 1998, 1997, 2010, 2013, 2017, 1975,
       1978, 1983, 1987, 2012, 2001, 2014, 2002, 2003, 2004, 2011, 2008,
       2009, 2007, 2005, 2006, 1994, 2015, 2019, 2016, 1982, 1989, 1990,
       1991, 1999, 1986, 1992, 1984, 1980, 1961, 2000, 1995, 1985, 1976,
       1959, 1988, 1981, 1972, 1964, 1945, 1954, 1979, 1958, 1956, 1963,
       1970, 1973, 1925, 1974, 1960, 1966, 1971, 1962, 1969, 1977, 1967,
       1968, 1965, 1946, 1942, 1955, 1944, 1947, 1943])

In [59]:
# Calculate how old the content is
current_year = datetime.now().year
df['content_age'] = current_year - df['release_year']

# Flag if content was added recently (after 2018)
df['is_recent_added'] = df['year_added'].apply(lambda x: 1 if x >= 2018 else 0)

# Flag classic content (before 2000)
df['is_classic'] = df['release_year'].apply(lambda x: 1 if x < 2000 else 0)

In [60]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,is_movie,cast_count,genre_count,country_count,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,1,1,1,1,United States,Documentaries,Kirsten Johnson,6,1,0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,0,19,3,1,South Africa,International TV Shows,Unknown,5,1,0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,0,9,3,1,Unknown,Crime TV Shows,Julien Leclercq,5,1,0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,0,1,2,1,Unknown,Docuseries,Unknown,5,1,0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,0,8,3,1,India,International TV Shows,Unknown,5,1,0


### 5.6 Rating Grouping

In [61]:
df['rating'].value_counts()

rating
TV-MA       3211
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64

In [62]:
# Group ratings into meaningful business categories
def rating_group(x):
    if x in ['G', 'PG', 'PG-13']:
        return 'Kids'
    elif x in ['R', 'NC-17']:
        return 'Adults'
    else:
        return 'General'

# Apply grouping
df['rating_group'] = df['rating'].apply(rating_group)

In [63]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,cast_count,genre_count,country_count,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,1,1,1,United States,Documentaries,Kirsten Johnson,6,1,0,Kids
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,19,3,1,South Africa,International TV Shows,Unknown,5,1,0,General
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,9,3,1,Unknown,Crime TV Shows,Julien Leclercq,5,1,0,General
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,1,2,1,Unknown,Docuseries,Unknown,5,1,0,General
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,8,3,1,India,International TV Shows,Unknown,5,1,0,General


### 6. Text Feature Engineering

### 6.1 Clean Description Text

In [64]:
df['description']

0       As her father nears the end of his life, filmm...
1       After crossing paths at a party, a Cape Town t...
2       To protect his family from a powerful drug lor...
3       Feuds, flirtations and toilet talk go down amo...
4       In a city of coaching centers known to train I...
                              ...                        
8802    A political cartoonist, a crime reporter and a...
8803    While living alone in a spooky town, a young g...
8804    Looking to survive in a world taken over by zo...
8805    Dragged from civilian life, a former superhero...
8806    A scrappy but poor boy worms his way into a ty...
Name: description, Length: 8807, dtype: str

In [65]:
# Function to clean text by:
# - converting to lowercase
# - removing punctuation
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)#[!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~]
    return text

    

# Apply cleaning
df['description_clean'] = df['description'].apply(clean_text)

In [66]:
#sub
#It replaces parts of a string that match a pattern with something else.
#re.sub(pattern, replacement, text)

In [67]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,genre_count,country_count,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,1,1,United States,Documentaries,Kirsten Johnson,6,1,0,Kids,as her father nears the end of his life filmma...
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,3,1,South Africa,International TV Shows,Unknown,5,1,0,General,after crossing paths at a party a cape town te...
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,3,1,Unknown,Crime TV Shows,Julien Leclercq,5,1,0,General,to protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,2,1,Unknown,Docuseries,Unknown,5,1,0,General,feuds flirtations and toilet talk go down amon...
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,3,1,India,International TV Shows,Unknown,5,1,0,General,in a city of coaching centers known to train i...


### 6.2 Description Length

In [68]:
# Count number of words in description
# Useful as a feature (long descriptions vs short)
df['desc_length'] = df['description'].apply(lambda x: len(x.split()))

In [69]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,country_count,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,1,United States,Documentaries,Kirsten Johnson,6,1,0,Kids,as her father nears the end of his life filmma...,27
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,1,South Africa,International TV Shows,Unknown,5,1,0,General,after crossing paths at a party a cape town te...,27
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,1,Unknown,Crime TV Shows,Julien Leclercq,5,1,0,General,to protect his family from a powerful drug lor...,27
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,1,Unknown,Docuseries,Unknown,5,1,0,General,feuds flirtations and toilet talk go down amon...,24
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,1,India,International TV Shows,Unknown,5,1,0,General,in a city of coaching centers known to train i...,24


In [70]:
df[df['desc_length']>10]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,country_count,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,1,United States,Documentaries,Kirsten Johnson,6,1,0,Kids,as her father nears the end of his life filmma...,27
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,1,South Africa,International TV Shows,Unknown,5,1,0,General,after crossing paths at a party a cape town te...,27
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,1,Unknown,Crime TV Shows,Julien Leclercq,5,1,0,General,to protect his family from a powerful drug lor...,27
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,1,Unknown,Docuseries,Unknown,5,1,0,General,feuds flirtations and toilet talk go down amon...,24
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,1,India,International TV Shows,Unknown,5,1,0,General,in a city of coaching centers known to train i...,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007,R,158 min,...,1,United States,Cult Movies,David Fincher,19,1,0,Adults,a political cartoonist a crime reporter and a ...,25
8803,s8804,TV Show,Zombie Dumb,Unknown,Unknown,Unknown,2019-07-01,2018,TV-Y7,2 Seasons,...,1,Unknown,Kids' TV,Unknown,8,1,0,General,while living alone in a spooky town a young gi...,20
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009,R,88 min,...,1,United States,Comedies,Ruben Fleischer,17,1,0,Adults,looking to survive in a world taken over by zo...,25
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006,PG,88 min,...,1,United States,Children & Family Movies,Peter Hewitt,20,1,0,Kids,dragged from civilian life a former superhero ...,26


# 7. Advanced Feature Engineering

In [71]:
df.columns


Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'year_added', 'month_added', 'day_added', 'duration_int', 'is_movie',
       'cast_count', 'genre_count', 'country_count', 'primary_country',
       'primary_genre', 'primary_director', 'content_age', 'is_recent_added',
       'is_classic', 'rating_group', 'description_clean', 'desc_length'],
      dtype='str')

### 7.1 Frequency Encoding

In [72]:
# Count how frequently each country appears
freq_country = df['primary_country'].value_counts()


#print(freq_country)
# Map frequency back to dataset
df['country_freq'] = df['primary_country'].map(freq_country)

In [73]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'year_added', 'month_added', 'day_added', 'duration_int', 'is_movie',
       'cast_count', 'genre_count', 'country_count', 'primary_country',
       'primary_genre', 'primary_director', 'content_age', 'is_recent_added',
       'is_classic', 'rating_group', 'description_clean', 'desc_length',
       'country_freq'],
      dtype='str')

In [74]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,primary_country,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length,country_freq
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,United States,Documentaries,Kirsten Johnson,6,1,0,Kids,as her father nears the end of his life filmma...,27,3211
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,South Africa,International TV Shows,Unknown,5,1,0,General,after crossing paths at a party a cape town te...,27,41
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,Unknown,Crime TV Shows,Julien Leclercq,5,1,0,General,to protect his family from a powerful drug lor...,27,831
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,Unknown,Docuseries,Unknown,5,1,0,General,feuds flirtations and toilet talk go down amon...,24,831
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,India,International TV Shows,Unknown,5,1,0,General,in a city of coaching centers known to train i...,24,1008


In [75]:
#identify dominant countries
#Analyze popularity distribution

### 7.2 Aggregation Feature

In [76]:
# Calculate average duration per genre
genre_duration = df.groupby('primary_genre')['duration_int'].mean()

print(genre_duration )
# Map this aggregated value back
df['genre_avg_duration'] = df['primary_genre'].map(genre_duration)

primary_genre
Action & Adventure              113.515716
Anime Features                   90.333333
Anime Series                      1.573864
British TV Shows                  1.818182
Children & Family Movies         78.426446
Classic & Cult TV                 5.727273
Classic Movies                  114.825000
Comedies                        106.687603
Crime TV Shows                    1.706767
Cult Movies                     102.500000
Documentaries                    82.149578
Docuseries                        1.366516
Dramas                          111.377500
Horror Movies                    98.174545
Independent Movies               98.700000
International Movies            108.062500
International TV Shows            1.339793
Kids' TV                          1.987113
LGBTQ Movies                     99.000000
Movies                           45.481481
Music & Musicals                 89.555556
Reality TV                        1.491667
Romantic Movies                  83.3333

In [77]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length,country_freq,genre_avg_duration
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,Documentaries,Kirsten Johnson,6,1,0,Kids,as her father nears the end of his life filmma...,27,3211,82.149578
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,International TV Shows,Unknown,5,1,0,General,after crossing paths at a party a cape town te...,27,41,1.339793
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,Crime TV Shows,Julien Leclercq,5,1,0,General,to protect his family from a powerful drug lor...,27,831,1.706767
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,Docuseries,Unknown,5,1,0,General,feuds flirtations and toilet talk go down amon...,24,831,1.366516
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,International TV Shows,Unknown,5,1,0,General,in a city of coaching centers known to train i...,24,1008,1.339793


In [78]:
# This is used to convert a categorical column (country) into a meaningful numeric feature based on its frequency.

# 8. Outlier Handling

In [79]:
# Calculate quartiles
Q1 = df['duration_int'].quantile(0.25)
Q3 = df['duration_int'].quantile(0.75)

# Interquartile range
IQR = Q3 - Q1

print(IQR)


104.0


In [80]:
ll=Q1 - 1.5 * IQR
ul=Q3 + 1.5 * IQR

print(ll)
print(ul)

-154.0
262.0


In [81]:
outliers = df[
    (df['duration_int'] < ll) |
    (df['duration_int'] > ul)
]

outliers

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length,country_freq,genre_avg_duration
717,s718,Movie,Headspace: Unwind Your Mind,Unknown,"Andy Puddicombe, Evelyn Lewis Prieto, Ginger D...",Unknown,2021-06-15,2021,TV-G,273 min,...,Documentaries,Unknown,5,1,0,General,do you want to relax meditate or sleep deeply ...,23,831,82.149578
4253,s4254,Movie,Black Mirror: Bandersnatch,Unknown,"Fionn Whitehead, Will Poulter, Craig Parkinson...",United States,2018-12-28,2018,TV-MA,312 min,...,Dramas,Unknown,8,1,0,General,in 1984 a young programmer begins to question ...,26,3211,111.377500


In [82]:
# q1=np.percentile(df['duration_int'],25)
# q3=np.percentile(df['duration_int'],75)

In [83]:
outliers[['duration_int']]

,duration_int
717,273.0
4253,312.0


In [84]:
# Remove extreme outliers
# df = df[
#     (df['duration_int'] >=ul) &
#      (df['duration_int'] <= ll)
#  ]

# 9. Feature Scaling

In [85]:
# Select numerical columns for scaling
num_cols = [
    'duration_int', 'cast_count', 'genre_count',
    'country_count', 'content_age', 'desc_length'
]

# Initialize scaler
sc = StandardScaler()

# Scale numerical features
df[num_cols] = sc.fit_transform(df[num_cols])

In [86]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,primary_genre,primary_director,content_age,is_recent_added,is_classic,rating_group,description_clean,desc_length,country_freq,genre_avg_duration
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,...,Documentaries,Kirsten Johnson,-0.659930,1,0,Kids,as her father nears the end of his life filmma...,1.099136,3211,82.149578
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,...,International TV Shows,Unknown,-0.773324,1,0,General,after crossing paths at a party a cape town te...,1.099136,41,1.339793
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,...,Crime TV Shows,Julien Leclercq,-0.773324,1,0,General,to protect his family from a powerful drug lor...,1.099136,831,1.706767
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,...,Docuseries,Unknown,-0.773324,1,0,General,feuds flirtations and toilet talk go down amon...,0.041511,831,1.366516
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,...,International TV Shows,Unknown,-0.773324,1,0,General,in a city of coaching centers known to train i...,0.041511,1008,1.339793


In [87]:
df[['duration_int', 'cast_count', 'genre_count','country_count', 'content_age', 'desc_length']].head()

,duration_int,cast_count,genre_count,country_count,content_age,desc_length
0,0.396622,-1.348169,-1.522448,-0.346862,-0.659930,1.099136
1,-1.335255,2.458469,1.027609,-0.346862,-0.773324,1.099136
2,-1.354935,0.343670,1.027609,-0.346862,-0.773324,1.099136
3,-1.354935,-1.348169,-0.247419,-0.346862,-0.773324,0.041511
4,-1.335255,0.132190,1.027609,-0.346862,-0.773324,0.041511


What is StandardScaler()?

StandardScaler is a tool from scikit-learn used to standardize numerical data

What does it do?




📊 After Scaling

Each feature will have:

✔ Mean = 0

✔ Standard Deviation = 1

🔥 Why do we use it? (VERY IMPORTANT)

✅ 1. Removes scale differences

Example:

Age → 25, 30

Salary → 50,000, 100,000

👉 Without scaling → salary dominates

👉 With scaling → equal importance

In [88]:
df.to_csv("netflix_final_ready.csv", index=False)